# Introducción (Fase 1)

El objetivo de esta práctica es construir un **MVP de reconocimiento de Lengua de Signos Española (LSE)** capaz de clasificar señas a partir de vídeo usando *landmarks* (puntos clave) extraídos con **MediaPipe**. En lugar de entrenar directamente con píxeles, representamos cada vídeo como una **secuencia temporal de coordenadas** (pose/manos), lo que reduce el coste computacional y facilita el aprendizaje de patrones de movimiento.

En esta **Fase 1** se implementa el *pipeline* completo de **generación de datos** para asegurar que la Fase 2 (entrenamiento del modelo) sea estable y reproducible. En concreto, el notebook cubre:

- **Setup y configuración del experimento** (vocabulario reducido y parámetros de captura).
- **Captura guiada con validación en tiempo real**, mostrando overlay de keypoints para asegurar buena detección.
- **Extracción automática de landmarks** por vídeo y almacenamiento en formato `.npy`.
- **Control de calidad (QC)** para filtrar muestras inconsistentes (por nº de landmarks/frames y estabilidad).
- **Data augmentation** (variaciones suaves) y **generación de splits** (train/val/test) coherentes.

El resultado final de esta fase es un dataset “limpio” y estructurado (metadata + landmarks) listo para entrenar un modelo secuencial (LSTM) y evaluar de forma justa en la Fase 2.


## 1. Setup del Entorno

Antes de comenzar, en esta primera celda se prepara el entorno de trabajo necesario para desarrollar toda la práctica.  
Se instalan e importan las librerías fundamentales para **captura de vídeo**, **extracción de landmarks**, **procesamiento de datos** y **análisis**.

- **OpenCV**  se utiliza para acceder a la cámara, capturar vídeo y mostrar la interfaz en tiempo real.
- **MediaPipe** proporciona los modelos de detección de pose y manos que permiten extraer los *landmarks* de la LSE.
- **NumPy** y **Pandas** se emplean para el manejo eficiente de arrays numéricos y metadatos.
- **Matplotlib** y **Seaborn** se usan para visualizar estadísticas y resultados del control de calidad.
- **Scikit-learn** se utiliza más adelante para generar los *splits* de entrenamiento, validación y test.
- **TQDM** permite mostrar barras de progreso durante procesos largos (captura, extracción o augmentación).
- Se incluyen utilidades estándar como `os`, `Path` y `datetime` para la gestión de archivos y nombres temporales.

Finalmente, se imprimen las versiones de OpenCV y MediaPipe para garantizar la **reproducibilidad del experimento** y facilitar la depuración en caso de incompatibilidades.


In [1]:
# Instalación de dependencias
# !pip install mediapipe opencv-python numpy pandas scikit-learn matplotlib seaborn tqdm -q

In [1]:
# Imports
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import os
import json
from datetime import datetime
import time
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

print("Bibliotecas importadas correctamente")
print(f"- OpenCV: {cv2.__version__}")
print(f"- MediaPipe: {mp.__version__}")

Bibliotecas importadas correctamente
- OpenCV: 4.11.0
- MediaPipe: 0.10.21


## 2. Configuración Optimizada para 10 Palabras

Definimos la **configuración global del experimento**, que controla tanto la captura de datos como los criterios de calidad y el formato final del dataset.  
Dado que el MVP se centra en un vocabulario reducido de **10 palabras**, se han ajustado los parámetros para priorizar **calidad y consistencia** frente a cantidad.

Aspectos clave de la configuración:

- **Vocabulario reducido (10 palabras)**: se seleccionan señas bien diferenciadas para facilitar el aprendizaje del modelo en un entorno controlado.
- **Más repeticiones por palabra** (`reps_per_word = 35`): se incrementa el número de muestras para mejorar la robustez estadística.
- **Captura de vídeo estable**: 3 segundos a 25 FPS y resolución 640×480, suficiente para describir el gesto completo.
- **Control de calidad (QC) estricto**:
  - Se exige un mínimo de **60 frames** por muestra.
  - Se diferencian umbrales de landmarks para señas de **una mano** (54) y **dos manos** (70).
  - Se eleva el umbral de calidad global al **92%**, reduciendo muestras ruidosas.
- **Extracción de solo manos** (`extract_hands_only = True`): se prioriza la información más relevante para la LSE, reduciendo dimensionalidad y ruido.
- **Opciones de visualización**: permiten mostrar landmarks y alertas de calidad durante la captura para facilitar la corrección en tiempo real.

Además, se define explícitamente qué palabras requieren **una o dos manos**, lo que permite adaptar tanto el control de calidad como el procesamiento posterior.  
La celda finaliza imprimiendo un resumen de la configuración y del vocabulario, asegurando transparencia y trazabilidad del experimento.


In [2]:
# Configuración global MEJORADA
CONFIG = {
    'project_name': 'sign_language_mvp_10words',
    'num_words': 10,
    'reps_per_word': 35,  
    'video_duration': 3,  # segundos
    'fps': 25,
    'resolution': (640, 480),
    
    # QC estricto para 10 palabras
    'min_frames': 60,  
    'min_landmarks_threshold': 70,  # Para 2 manos
    'min_landmarks_threshold_one_hand': 54,  # Para 1 mano
    'quality_threshold': 0.92, 
    
    'signer_id': 0,
    'max_seq_length': 100,
    
    # Configuración de visualización
    'show_landmarks': True,
    'show_quality_alerts': True,
    
    # Extracción de solo manos
    'extract_hands_only': True,
}

# Vocabulario utilizado
VOCABULARY = [
    'HOLA', 'GRACIAS', 'POR_FAVOR',
    'SI', 'NO',
    'YO', 'TU',
    'AYUDA', 'BANIO', 'COMER'
]

# Mapeo de signos de 1/2 manos
TWO_HAND_SIGNS = {'GRACIAS', 'AYUDA'}
ONE_HAND_SIGNS = set(VOCABULARY) - TWO_HAND_SIGNS

print("=" * 60)
print("CONFIGURACIÓN MEJORADA PARA 10 PALABRAS")
print("=" * 60)
print(f"Vocabulario: {CONFIG['num_words']} palabras")
print(f"Repeticiones por palabra: {CONFIG['reps_per_word']}")
print(f"Videos objetivo: {CONFIG['num_words'] * CONFIG['reps_per_word']}")
print(f"Quality threshold: {CONFIG['quality_threshold']}")
print(f"\nPalabras:")
for i, word in enumerate(VOCABULARY, 1):
    hand_type = "2 manos" if word in TWO_HAND_SIGNS else "1 mano"
    print(f"  {i:2d}. {word:15s} ({hand_type})")
print("=" * 60)

CONFIGURACIÓN MEJORADA PARA 10 PALABRAS
Vocabulario: 10 palabras
Repeticiones por palabra: 35
Videos objetivo: 350
Quality threshold: 0.92

Palabras:
   1. HOLA            (1 mano)
   2. GRACIAS         (2 manos)
   3. POR_FAVOR       (1 mano)
   4. SI              (1 mano)
   5. NO              (1 mano)
   6. YO              (1 mano)
   7. TU              (1 mano)
   8. AYUDA           (2 manos)
   9. BANIO           (1 mano)
  10. COMER           (1 mano)


Creamos la **estructura de directorios** necesaria para organizar de forma clara y reproducible todos los artefactos del proyecto.  

La estructura generada incluye:

- **`data/raw_videos/`**: vídeos originales capturados desde la cámara, organizados por palabra.
- **`data/landmarks/`**: secuencias de landmarks extraídas de cada vídeo (formato `.npy`).
- **`data/landmarks_hands_only/`**: versión alternativa del dataset que contiene únicamente landmarks de las manos (si está activado).
- **`data/metadata/`**: ficheros CSV con metadatos, control de calidad y splits del dataset.
- **`models/`**: modelos entrenados y checkpoints.
- **`outputs/reports/` y `outputs/visualizations/`**: métricas, gráficos y resultados finales.

Además, para cada palabra del vocabulario se crean subcarpetas específicas dentro de `raw_videos`, `landmarks` y `landmarks_hands_only`, facilitando el acceso y el procesamiento por clase.

Por último, el vocabulario utilizado en el experimento se guarda en un fichero de texto (`vocabulario_10_palabras.txt`), lo que permite **documentar y reutilizar** fácilmente la lista de señas en fases posteriores (entrenamiento, inferencia o despliegue).


In [3]:
# Crear estructura de directorios
def create_directory_structure():
    dirs = [
        'data/raw_videos', 'data/landmarks', 'data/landmarks_hands_only',
        'data/metadata', 'models', 'outputs/reports', 'outputs/visualizations'
    ]
    
    for dir_path in dirs:
        Path(dir_path).mkdir(parents=True, exist_ok=True)
    
    for word in VOCABULARY:
        Path(f'data/raw_videos/{word}').mkdir(exist_ok=True)
        Path(f'data/landmarks/{word}').mkdir(exist_ok=True)
        if CONFIG['extract_hands_only']:
            Path(f'data/landmarks_hands_only/{word}').mkdir(exist_ok=True)
    
    print("Estructura de directorios creada")

create_directory_structure()

with open('vocabulario_10_palabras.txt', 'w') as f:
    f.write('\n'.join(VOCABULARY))
print("Vocabulario guardado")

Estructura de directorios creada
Vocabulario guardado


## 3. Sistema de Captura con Overlay de Keypoints

Sse define la clase **`ManualDataCollector`**, responsable de la **captura guiada de vídeos** y de asegurar que las muestras grabadas tengan la calidad suficiente antes de incorporarse al dataset.

La clase se apoya en **MediaPipe Holistic** para detectar *landmarks* de pose y manos, y añade varias mejoras clave respecto a una captura básica:

- **Extracción consistente de landmarks por frame**:  
  Cada fotograma se representa siempre con el mismo número de puntos (33 de pose + 21 por mano), rellenando con ceros cuando no se detectan landmarks. Esto garantiza homogeneidad en los datos.

- **Cálculo de calidad adaptativo**:  
  El método `calculate_quality_adaptive` evalúa cada vídeo en función del **tipo de seña** (una o dos manos), aplicando distintos umbrales de landmarks. Así se evitan rechazos injustificados en señas de una sola mano.

- **Overlay visual de landmarks y feedback inmediato**:  
  Durante la captura se muestran:
  - Landmarks de pose y manos con colores diferenciados.
  - Indicadores visuales de detección de manos (izquierda/derecha).
  - Barra de progreso de calidad y alertas cuando la seña no cumple los requisitos.
  Esto permite al usuario corregir la ejecución **en tiempo real**.

- **Grabación controlada por número de frames**:  
  En lugar de grabar por tiempo fijo, cada vídeo se captura hasta alcanzar un número objetivo de frames (por ejemplo, 60), asegurando secuencias temporales uniformes y alineadas con el entrenamiento posterior.

- **Validación automática tras cada repetición**:  
  Una repetición solo se acepta si supera el umbral de calidad configurado; en caso contrario, se descarta y se solicita una nueva grabación.

- **Captura completa del dataset**:  
  El método `capture_full_dataset` automatiza la grabación de todas las palabras del vocabulario, guardando además un *log* con metadatos (palabra, calidad, número de frames, timestamp).

In [3]:
class ManualDataCollector:
    """Sistema de captura mejorado con overlay de keypoints y feedback visual."""
    
    def __init__(self, config):
        self.config = config
        self.mp_holistic = mp.solutions.holistic.Holistic(
            static_image_mode=False, model_complexity=0,
            smooth_landmarks=True, min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        )
        self.mp_drawing = mp.solutions.drawing_utils
        self.mp_drawing_styles = mp.solutions.drawing_styles
        self.capture_log = []
    
    def extract_landmarks_frame(self, results):
        landmarks = []
        if results.pose_landmarks:
            landmarks.extend([[lm.x, lm.y, lm.z] for lm in results.pose_landmarks.landmark])
        else:
            landmarks.extend([[0, 0, 0]] * 33)
        
        for hand in [results.left_hand_landmarks, results.right_hand_landmarks]:
            if hand:
                landmarks.extend([[lm.x, lm.y, lm.z] for lm in hand.landmark])
            else:
                landmarks.extend([[0, 0, 0]] * 21)
        
        return np.array(landmarks)
    
    def calculate_quality_adaptive(self, landmarks_seq, word):
        if len(landmarks_seq) == 0:
            return 0.0
        
        min_required = self.config['min_landmarks_threshold'] if word in TWO_HAND_SIGNS \
                      else self.config['min_landmarks_threshold_one_hand']
        
        valid_frames = 0
        for landmarks in landmarks_seq:
            pose_nz = np.sum(np.any(landmarks[:33] != 0, axis=1))
            left_nz = np.sum(np.any(landmarks[33:54] != 0, axis=1))
            right_nz = np.sum(np.any(landmarks[54:] != 0, axis=1))
            if (pose_nz + left_nz + right_nz) >= min_required:
                valid_frames += 1
        
        return valid_frames / len(landmarks_seq)
    
    def draw_landmarks_overlay(self, frame, results):
        if not self.config['show_landmarks']:
            return frame
        
        if results.pose_landmarks:
            self.mp_drawing.draw_landmarks(
                frame, results.pose_landmarks, mp.solutions.holistic.POSE_CONNECTIONS,
                landmark_drawing_spec=self.mp_drawing_styles.get_default_pose_landmarks_style()
            )
        
        if results.left_hand_landmarks:
            self.mp_drawing.draw_landmarks(
                frame, results.left_hand_landmarks, mp.solutions.hands.HAND_CONNECTIONS,
                self.mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=2),
                self.mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2)
            )
        
        if results.right_hand_landmarks:
            self.mp_drawing.draw_landmarks(
                frame, results.right_hand_landmarks, mp.solutions.hands.HAND_CONNECTIONS,
                self.mp_drawing.DrawingSpec(color=(255, 0, 0), thickness=2, circle_radius=2),
                self.mp_drawing.DrawingSpec(color=(255, 0, 0), thickness=2)
            )
        
        return frame
    
    def draw_feedback_improved(self, frame, results, word, rep_num, quality, is_recording=False):
        h, w, _ = frame.shape
        header_color = (50, 50, 50) if not is_recording else (0, 0, 139)
        cv2.rectangle(frame, (0, 0), (w, 100), header_color, -1)
        
        cv2.putText(frame, f"Palabra: {word}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.putText(frame, f"Rep: {rep_num + 1}/{self.config['reps_per_word']}", (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 2)
        
        if is_recording:
            cv2.circle(frame, (w - 40, 30), 15, (0, 0, 255), -1)
            cv2.putText(frame, "REC", (w - 130, 35),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
        
        left_detected = results.left_hand_landmarks is not None
        right_detected = results.right_hand_landmarks is not None
        
        left_color = (0, 255, 0) if left_detected else (0, 0, 255)
        right_color = (0, 255, 0) if right_detected else (0, 0, 255)
        
        cv2.rectangle(frame, (w - 180, 50), (w - 150, 80), left_color, -1)
        cv2.putText(frame, "L", (w - 172, 72), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        
        cv2.rectangle(frame, (w - 140, 50), (w - 110, 80), right_color, -1)
        cv2.putText(frame, "R", (w - 132, 72), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        
        if quality > 0:
            quality_color = (0, 255, 0) if quality >= self.config['quality_threshold'] else \
                           (0, 165, 255) if quality >= 0.85 else (0, 0, 255)
            
            cv2.putText(frame, f"Calidad: {quality:.0%}", (10, h - 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, quality_color, 2)
            
            bar_width = int(300 * quality)
            cv2.rectangle(frame, (10, h - 40), (310, h - 20), (50, 50, 50), -1)
            cv2.rectangle(frame, (10, h - 40), (10 + bar_width, h - 20), quality_color, -1)
            cv2.rectangle(frame, (10, h - 40), (310, h - 20), (255, 255, 255), 2)
            
            threshold_x = int(10 + 300 * self.config['quality_threshold'])
            cv2.line(frame, (threshold_x, h - 45), (threshold_x, h - 15), (255, 255, 0), 2)
        
        if self.config['show_quality_alerts'] and is_recording:
            alerts = []
            if word in TWO_HAND_SIGNS and not (left_detected and right_detected):
                alerts.append("¡Se necesitan 2 manos!")
            if not left_detected and not right_detected:
                alerts.append("¡No se detectan manos!")
            if quality > 0 and quality < 0.85:
                alerts.append("Calidad baja - manos en el cuadro")
            
            for i, alert in enumerate(alerts):
                cv2.putText(frame, f"⚠ {alert}", (10, 120 + i * 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)
        
        cv2.putText(frame, "'q' saltar | ESC cancelar", (10, h - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
        
        return frame
    
    def capture_word(self, word, num_reps=None, target_frames=60):
        """
        Captura múltiples repeticiones de una palabra (ImprovedManualDataCollector)
        - Graba hasta alcanzar `target_frames` (p.ej. 60) en vez de por tiempo.
        - Usa calculate_quality_adaptive(...), draw_landmarks_overlay(...), draw_feedback_improved(...).
        - Escribe al vídeo el frame "limpio" (sin overlays) para reducir carga.
        """

        if num_reps is None:
            num_reps = self.config['reps_per_word']

        # Asegurar carpeta destino
        os.makedirs(f"data/raw_videos/{word}", exist_ok=True)

        cap = cv2.VideoCapture(0)
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, self.config['resolution'][0])
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, self.config['resolution'][1])
        cap.set(cv2.CAP_PROP_FPS, self.config['fps'])

        if not cap.isOpened():
            print("❌ Error: No se puede acceder a la cámara")
            return

        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        successful_captures = 0
        quality_scores = []

        print(f"\n{'='*60}")
        print(f"🎬 CAPTURANDO: {word}")
        print(f"{'='*60}")
        print(f"Objetivo: {num_reps} repeticiones válidas")
        print(f"Frames objetivo por vídeo: {target_frames}")
        print("Presiona 'q' para saltar, 'ESC' para cancelar\n")

        while successful_captures < num_reps:

            # --- Countdown 3-2-1 ---
            for i in range(3, 0, -1):
                ret, frame = cap.read()
                if not ret:
                    break

                frame = cv2.flip(frame, 1)
                cv2.putText(frame, f'{i}',
                            (frame.shape[1] // 2 - 50, frame.shape[0] // 2),
                            cv2.FONT_HERSHEY_SIMPLEX, 5, (0, 255, 0), 10)
                cv2.putText(frame, f"{word}  Rep {successful_captures+1}/{num_reps}",
                            (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
                cv2.imshow('Captura LSE - Manual MEJORADO', frame)

                key = cv2.waitKey(1000)
                if key == 27:  # ESC
                    cap.release()
                    cv2.destroyAllWindows()
                    return

            # --- Preparar grabación ---
            timestamp = datetime.now().strftime("%H%M%S")
            video_path = f"data/raw_videos/{word}/{word}_rep_{successful_captures}_{timestamp}.mp4"

            # Leer un frame para asegurar el tamaño real (evita errores de VideoWriter)
            ret, test_frame = cap.read()
            if not ret:
                print("❌ No se pudo leer frame para inicializar VideoWriter.")
                break
            test_frame = cv2.flip(test_frame, 1)
            h, w = test_frame.shape[:2]

            out = cv2.VideoWriter(video_path, fourcc, self.config['fps'], (w, h))
            if not out.isOpened():
                print("❌ Error: VideoWriter no se pudo abrir. Revisa codec/resolución.")
                continue

            landmarks_seq = []
            t0 = time.perf_counter()

            # --- Grabación: por objetivo de frames ---
            while len(landmarks_seq) < target_frames:
                ret, frame = cap.read()
                if not ret:
                    break

                frame = cv2.flip(frame, 1)
                rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                results = self.mp_holistic.process(rgb_frame)

                # Extraer landmarks
                frame_landmarks = self.extract_landmarks_frame(results)
                landmarks_seq.append(frame_landmarks)

                # Calidad parcial (adaptive -> depende de la palabra / 1 mano vs 2 manos)
                partial_quality = self.calculate_quality_adaptive(landmarks_seq, word)

                # Overlay landmarks + feedback (solo para visualizar)
                frame_vis = self.draw_landmarks_overlay(frame.copy(), results)
                frame_vis = self.draw_feedback_improved(
                    frame_vis, results, word, successful_captures, partial_quality, is_recording=True
                )

                # Guardar en vídeo el frame "limpio" (menos carga y menos compresión)
                if frame.shape[1] != w or frame.shape[0] != h:
                    frame_to_write = cv2.resize(frame, (w, h))
                else:
                    frame_to_write = frame
                out.write(frame_to_write)

                cv2.imshow('Captura LSE - Manual MEJORADO', frame_vis)

                key = cv2.waitKey(1)
                if key == 27:  # ESC
                    out.release()
                    cap.release()
                    cv2.destroyAllWindows()
                    return
                if key == ord('q'):
                    # Salta esta repetición (se evaluará y probablemente se descartará)
                    break

            out.release()

            elapsed = time.perf_counter() - t0
            real_fps = len(landmarks_seq) / max(elapsed, 1e-6)
            print(f"ℹ️  Grabación: {len(landmarks_seq)} frames | FPS real ~ {real_fps:.1f}")

            # --- Validación final ---
            final_quality = self.calculate_quality_adaptive(landmarks_seq, word)
            quality_scores.append(final_quality)

            if final_quality >= self.config['quality_threshold'] and len(landmarks_seq) >= target_frames:
                print(f"✅ Rep {successful_captures + 1}/{num_reps} - Calidad: {final_quality:.2%} - ACEPTADA")

                self.capture_log.append({
                    'word': word,
                    'file_path': video_path,
                    'num_frames': len(landmarks_seq),
                    'quality_score': final_quality,
                    'signer_id': self.config['signer_id'],
                    'timestamp': timestamp
                })

                successful_captures += 1
            else:
                reasons = []
                if len(landmarks_seq) < target_frames:
                    reasons.append(f"frames {len(landmarks_seq)} < {target_frames}")
                if final_quality < self.config['quality_threshold']:
                    reasons.append(f"calidad {final_quality:.2%} < {self.config['quality_threshold']:.0%}")

                print(f"❌ RECHAZADA ({', '.join(reasons)}) - RECAPTURANDO...")
                try:
                    os.remove(video_path)
                except Exception:
                    pass

            cv2.waitKey(350)

        cap.release()
        cv2.destroyAllWindows()

        avg_quality = float(np.mean(quality_scores)) if quality_scores else 0.0
        print(f"\n✅ {word} completado - Calidad promedio: {avg_quality:.2%}\n")
        return avg_quality


    def capture_full_dataset(self, word_list=None):
        if word_list is None:
            word_list = VOCABULARY
        
        print("\n" + "#" * 60)
        print("#  CAPTURA DE DATASET COMPLETO (10 PALABRAS)  ".center(60))
        print("#" * 60)
        
        word_qualities = {}
        
        for i, word in enumerate(word_list, 1):
            print(f"\n[{i}/{len(word_list)}] 📹 Capturando: {word}")
            avg_quality = self.capture_word(word)
            word_qualities[word] = avg_quality
            
            pd.DataFrame(self.capture_log).to_csv('data/metadata/capture_log.csv', index=False)
        
        print("\n" + "#" * 60)
        print("#  ✅ CAPTURA COMPLETADA  ".center(60))
        print("#" * 60)
        print(f"Videos capturados: {len(self.capture_log)}")
        print(f"Calidad promedio: {np.mean(list(word_qualities.values())):.0%}")
        
        return word_qualities

print("Clase ManualDataCollector definida")

Clase ManualDataCollector definida


## 4. Test de Cámara

Implementamos una **prueba rápida de la cámara** con overlay de *keypoints*, cuyo objetivo es verificar que **MediaPipe detecta correctamente la pose y las manos** antes de iniciar la captura del dataset.

Durante la ejecución:
- Se capturan hasta **200 frames** desde la webcam.
- Se dibujan en pantalla los *landmarks* de:
  - **Pose** (en estilo estándar de MediaPipe).
  - **Mano izquierda** (en verde).
  - **Mano derecha** (en azul/rojo).
- El vídeo se muestra en tiempo real con efecto espejo para facilitar la interacción.

Además de la visualización, se calcula una **tasa de detección** basada en cuántos frames contienen manos detectadas. Al finalizar, el sistema informa si la detección es:
- **Óptima**  
- **Mejorable** 
- **Insuficiente**

según el porcentaje de detecciones exitosas.

Este test es un paso previo muy importante, ya que nos permite comprobar condiciones de **iluminación, encuadre y estabilidad de la cámara**, evitando problemas de calidad antes de grabar las muestras definitivas del dataset.

In [4]:
def test_camera_with_overlay():
    print("\n📹 Test de cámara con overlay de keypoints")
    print("Presiona 'q' para salir\n")
    
    cap = cv2.VideoCapture(0)
    mp_holistic = mp.solutions.holistic.Holistic()
    mp_drawing = mp.solutions.drawing_utils
    mp_drawing_styles = mp.solutions.drawing_styles
    
    frame_count = 0
    detection_count = 0
    
    while frame_count < 200:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame = cv2.flip(frame, 1)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = mp_holistic.process(rgb_frame)
        
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(
                frame, results.pose_landmarks, mp.solutions.holistic.POSE_CONNECTIONS,
                landmark_drawing_spec=mp_drawing_styles.get_default_pose_landmarks_style()
            )
        
        if results.left_hand_landmarks:
            mp_drawing.draw_landmarks(
                frame, results.left_hand_landmarks, mp.solutions.hands.HAND_CONNECTIONS,
                mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=2),
                mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2)
            )
            detection_count += 1
        
        if results.right_hand_landmarks:
            mp_drawing.draw_landmarks(
                frame, results.right_hand_landmarks, mp.solutions.hands.HAND_CONNECTIONS,
                mp_drawing.DrawingSpec(color=(255, 0, 0), thickness=2, circle_radius=2),
                mp_drawing.DrawingSpec(color=(255, 0, 0), thickness=2)
            )
            detection_count += 1
        
        cv2.putText(frame, "Test - Presiona 'q' para salir", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        
        cv2.imshow('Test Camara', frame)
        
        frame_count += 1
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()
    
    rate = detection_count / (frame_count * 2) if frame_count > 0 else 0
    print(f"\n{'='*60}")
    print(f"Tasa de detección: {rate:.0%}")
    print(f"Estado: {'✅ ÓPTIMO' if rate >= 0.7 else '⚠ MEJORABLE' if rate >= 0.4 else '❌ INSUFICIENTE'}")
    print(f"{'='*60}")

# Descomenta para ejecutar test
test_camera_with_overlay()


📹 Test de cámara con overlay de keypoints
Presiona 'q' para salir



I0000 00:00:1767960341.174741 9422032 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1767960341.268441 9423491 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1767960341.280415 9423490 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1767960341.283690 9423488 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1767960341.283771 9423491 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1767960341.285677 9423490 inference_feedback_manager.cc:114] Feedback manager requires a mod


Tasa de detección: 85%
Estado: ✅ ÓPTIMO


## 5. Captura del Dataset

Inicializamos el **colector manual**.

En lugar de grabar todo el dataset de una sola vez, se ha seguido una **estrategia incremental palabra a palabra**, lo que aporta mayor control sobre la calidad y diversidad de las muestras.

La idea principal es que, para cada palabra:
- Se graban **varias repeticiones**, pudiendo ajustar el número según la dificultad de la seña.
- En cada repetición (o grupo de repeticiones) se **varía deliberadamente la ejecución**:
  - Cambios en la **postura corporal**.
  - Diferente **distancia a la cámara**.
  - Variaciones en el **ángulo y orientación de las manos**.
  - Pequeñas diferencias en la velocidad o amplitud del gesto.

Esta estrategia nos permite generar un dataset **más diverso y representativo**, evitando que el modelo aprenda únicamente una única forma “ideal” de realizar cada seña.  
Como consecuencia, el modelo entrenado en fases posteriores generaliza mejor ante variaciones reales durante la inferencia en tiempo real.

In [5]:
# Inicializar colector
collector = ManualDataCollector(CONFIG)

# Grabamos palabra a palabra variando el número de repeticiones
collector.capture_word('AYUDA', num_reps=4)

print("Colector inicializado")

I0000 00:00:1767960978.169216 9422032 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3
W0000 00:00:1767960978.254792 9432214 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1767960978.266337 9432214 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1767960978.269518 9432217 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1767960978.270515 9432214 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1767960978.270573 9432221 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support 


🎬 CAPTURANDO: AYUDA
Objetivo: 4 repeticiones válidas
Frames objetivo por vídeo: 60
Presiona 'q' para saltar, 'ESC' para cancelar

ℹ️  Grabación: 60 frames | FPS real ~ 22.0
❌ RECHAZADA (calidad 0.00% < 92%) - RECAPTURANDO...
ℹ️  Grabación: 60 frames | FPS real ~ 22.6
❌ RECHAZADA (calidad 5.00% < 92%) - RECAPTURANDO...
ℹ️  Grabación: 60 frames | FPS real ~ 21.9
❌ RECHAZADA (calidad 0.00% < 92%) - RECAPTURANDO...
ℹ️  Grabación: 60 frames | FPS real ~ 21.2
❌ RECHAZADA (calidad 80.00% < 92%) - RECAPTURANDO...
ℹ️  Grabación: 60 frames | FPS real ~ 20.3
✅ Rep 1/4 - Calidad: 100.00% - ACEPTADA
ℹ️  Grabación: 60 frames | FPS real ~ 19.7
✅ Rep 2/4 - Calidad: 100.00% - ACEPTADA
ℹ️  Grabación: 60 frames | FPS real ~ 20.3
✅ Rep 3/4 - Calidad: 100.00% - ACEPTADA
ℹ️  Grabación: 60 frames | FPS real ~ 21.6
✅ Rep 4/4 - Calidad: 100.00% - ACEPTADA

✅ AYUDA completado - Calidad promedio: 60.62%

Colector inicializado


## 6. Extracción de Landmarks

En esta fase transformamos cada vídeo grabado en una **secuencia numérica** lista para entrenar el modelo. La clase `LandmarkExtractor` automatiza el proceso completo:

- **Inicialización (MediaPipe Holistic):** se crea un detector `Holistic` con parámetros estables (`model_complexity=0`, *smoothing*, y umbrales de detección/tracking). Esto asegura consistencia con la fase de captura.

- **Resampleo temporal a longitud fija:** como cada vídeo puede tener un número distinto de fotogramas, se re-muestrea la secuencia para obtener siempre `target_frames=60`.  
  - `_resample_to_len(...)` reescala secuencias 2D/3D (T, …)  
  - `_resample_1d(...)` reescala secuencias 1D (por ejemplo, nº de manos detectadas por frame)

- **Construcción del vector de landmarks por frame:** en `extract_from_video(...)` se procesa cada fotograma:
  - se aplica `cv2.flip(frame, 1)` para mantener la misma orientación que en la captura
  - se extraen **75 puntos** en total: **33 de pose** + **21 mano izquierda** + **21 mano derecha**
  - si alguna parte no se detecta, se rellena con ceros para mantener dimensiones constantes

- **Normalización espacial** `normalize_landmarks(...)` centra las coordenadas usando el punto medio entre hombros (pose landmarks 11 y 12) y escala por la distancia entre hombros.  
  Esto hace el modelo más robusto a **distancia a cámara** y **posición del cuerpo**.

- **Modo “hands-only”:** `extract_hands_only(...)` permite quedarnos solo con manos (42 puntos) para reducir ruido del cuerpo y simplificar el input.

- **Procesado del dataset completo:** `process_dataset(...)` recorre el directorio `data/raw_videos/{PALABRA}`:
  - extrae y guarda `.npy` con landmarks completos (y opcionalmente hands-only)
  - calcula métricas de calidad por vídeo: nº medio de landmarks no nulos, ratio de frames válidos, media de manos detectadas, etc.
  - guarda todo en `data/metadata/landmarks_metadata.csv` para auditoría y filtrado posterior

Finalmente, al ejecutar:
- `extractor = LandmarkExtractor(CONFIG)`
- `df_landmarks = extractor.process_dataset()`

se obtiene un **DataFrame resumen** y se crean los ficheros `.npy` que alimentarán la fase de entrenamiento.

In [ ]:
class LandmarkExtractor:
    def __init__(self, config, target_frames=60):
        self.config = config
        self.target_frames = target_frames

        self.normalize = self.config.get('normalize_landmarks', True)

        # Holistic alineado con captura (estable)
        self.mp_holistic = mp.solutions.holistic.Holistic(
            static_image_mode=False,
            model_complexity=0,
            smooth_landmarks=True,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        )

    # Utils
    @staticmethod
    def _resample_to_len(seq, target_len=60):
        """Resample genérico para arrays (T,...)"""
        T = len(seq)
        if T == 0:
            return np.zeros((target_len,) + seq.shape[1:], dtype=np.float32)
        idx = np.linspace(0, T - 1, target_len).astype(int)
        return seq[idx]

    @staticmethod
    def _resample_1d(seq, target_len):
        """Resample limpio para arrays 1D"""
        T = len(seq)
        if T == 0:
            return np.zeros((target_len,), dtype=np.int32)
        idx = np.linspace(0, T - 1, target_len).astype(int)
        return seq[idx]

    @staticmethod
    def _frame_counts(lm75):
        pose_nz = np.sum(np.any(lm75[:33] != 0, axis=1))
        left_nz = np.sum(np.any(lm75[33:54] != 0, axis=1))
        right_nz = np.sum(np.any(lm75[54:75] != 0, axis=1))
        hands_present = int(left_nz > 0) + int(right_nz > 0)
        non_zero = pose_nz + left_nz + right_nz
        return non_zero, hands_present

    
    # Normalización espacial 
    @staticmethod
    def normalize_landmarks(landmarks):
        """
        Centra por hombros y escala por distancia entre hombros.
        landmarks: (T,75,3)
        """
        L = landmarks.copy()

        # Pose shoulders: 11 (izq), 12 (der)
        left_sh = L[:, 11, :2]
        right_sh = L[:, 12, :2]

        center = (left_sh + right_sh) / 2.0
        scale = np.linalg.norm(left_sh - right_sh, axis=1, keepdims=True)
        scale[scale < 1e-6] = 1.0

        for t in range(L.shape[0]):
            for j in range(L.shape[1]):
                if np.any(L[t, j, :] != 0):
                    L[t, j, 0] = (L[t, j, 0] - center[t, 0]) / scale[t, 0]
                    L[t, j, 1] = (L[t, j, 1] - center[t, 1]) / scale[t, 0]

        return L.astype(np.float32)

    # Extracción
    def extract_from_video(self, video_path):
        cap = cv2.VideoCapture(video_path)

        landmarks_sequence = []
        hands_count_sequence = []

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            # Consistencia con captura
            frame = cv2.flip(frame, 1)

            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = self.mp_holistic.process(rgb_frame)

            hands_detected = int(results.left_hand_landmarks is not None) + \
                             int(results.right_hand_landmarks is not None)
            hands_count_sequence.append(hands_detected)

            frame_landmarks = []

            # Pose (33)
            if results.pose_landmarks:
                frame_landmarks.extend([[lm.x, lm.y, lm.z] for lm in results.pose_landmarks.landmark])
            else:
                frame_landmarks.extend([[0, 0, 0]] * 33)

            # Hands (21 + 21)
            for hand in [results.left_hand_landmarks, results.right_hand_landmarks]:
                if hand:
                    frame_landmarks.extend([[lm.x, lm.y, lm.z] for lm in hand.landmark])
                else:
                    frame_landmarks.extend([[0, 0, 0]] * 21)

            landmarks_sequence.append(frame_landmarks)

        cap.release()

        if len(landmarks_sequence) == 0:
            return (
                np.zeros((0, 75, 3), dtype=np.float32),
                np.zeros((0,), dtype=np.int32)
            )

        landmarks = np.array(landmarks_sequence, dtype=np.float32)
        hands_counts = np.array(hands_count_sequence, dtype=np.int32)

        # Resample fijo
        landmarks = self._resample_to_len(landmarks, self.target_frames)
        hands_counts = self._resample_1d(hands_counts, self.target_frames)

        # Normalización espacial 
        if self.normalize:
            landmarks = self.normalize_landmarks(landmarks)

        return landmarks, hands_counts

    # Hands-only
    def extract_hands_only(self, landmarks_75):
        left_hand = landmarks_75[:, 33:54, :]
        right_hand = landmarks_75[:, 54:75, :]
        return np.concatenate([left_hand, right_hand], axis=1)  

    # Dataset
    def process_dataset(self,
                        raw_videos_dir='data/raw_videos',
                        landmarks_dir='data/landmarks',
                        hands_only_dir='data/landmarks_hands_only'):
        print("\nExtrayendo landmarks...")

        metadata = []
        total_videos = 0

        for word in tqdm(VOCABULARY, desc="Procesando"):
            word_video_path = Path(raw_videos_dir) / word
            if not word_video_path.exists():
                continue

            word_landmark_path = Path(landmarks_dir) / word
            word_landmark_path.mkdir(parents=True, exist_ok=True)

            word_hands_path = Path(hands_only_dir) / word
            if self.config.get('extract_hands_only', False):
                word_hands_path.mkdir(parents=True, exist_ok=True)

            for video_file in word_video_path.glob('*.mp4'):
                landmarks, hands_counts = self.extract_from_video(str(video_file))
                if len(landmarks) == 0:
                    continue

                npy_name = video_file.stem + '.npy'
                npy_path = word_landmark_path / npy_name
                np.save(npy_path, landmarks)

                hands_npy_path = None
                if self.config.get('extract_hands_only', False):
                    hands_only = self.extract_hands_only(landmarks)
                    hands_npy_path = word_hands_path / npy_name
                    np.save(hands_npy_path, hands_only)

                # Métricas
                nonzero_counts = []
                valid_frames = 0
                min_two = CONFIG.get('min_landmarks_threshold', 70)
                min_one = CONFIG.get('min_landmarks_threshold_one_hand', 54)

                for lm, hc in zip(landmarks, hands_counts):
                    non_zero, hands_present = self._frame_counts(lm)
                    nonzero_counts.append(non_zero)
                    required = min_two if hands_present == 2 else min_one
                    if non_zero >= required:
                        valid_frames += 1

                metadata.append({
                    'word': word,
                    'video_file': str(video_file),
                    'landmarks_file': str(npy_path),
                    'landmarks_hands_only_file': str(hands_npy_path) if hands_npy_path else None,
                    'num_frames': int(len(landmarks)),
                    'avg_landmarks': float(np.mean(nonzero_counts)),
                    'valid_frame_ratio': float(valid_frames / len(landmarks)),
                    'signer_id': CONFIG['signer_id'],
                    'avg_hands_detected': float(np.mean(hands_counts)),
                    'both_hands_ratio': float(np.mean(hands_counts == 2)),
                    'one_hand_ratio': float(np.mean(hands_counts == 1)),
                    'no_hand_ratio': float(np.mean(hands_counts == 0)),
                })

                total_videos += 1

        df_metadata = pd.DataFrame(metadata)
        Path("data/metadata").mkdir(parents=True, exist_ok=True)
        df_metadata.to_csv('data/metadata/landmarks_metadata.csv', index=False)

        print(f"\nExtracción completada: {total_videos} vídeos")
        print(f"Secuencias normalizadas a {self.target_frames} frames")

        return df_metadata



# Ejecutar extracción (descomenta)
extractor = LandmarkExtractor(CONFIG)
df_landmarks = extractor.process_dataset()
print(df_landmarks)
print("Extractor definido")

I0000 00:00:1767779590.653738 8584769 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3



🔄 Extrayendo landmarks...


Procesando:   0%|          | 0/10 [00:00<?, ?it/s]W0000 00:00:1767779590.748223 8604993 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1767779590.760612 8604996 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1767779590.765393 8604997 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1767779590.765543 8604996 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1767779590.770532 8604998 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1767779590.771098 8604996 in


✅ Extracción completada: 398 vídeos
ℹ️  Secuencias normalizadas a 60 frames
      word                                    video_file  \
0     HOLA    data/raw_videos/HOLA/HOLA_rep_2_114458.mp4   
1     HOLA    data/raw_videos/HOLA/HOLA_rep_1_121412.mp4   
2     HOLA    data/raw_videos/HOLA/HOLA_rep_1_110156.mp4   
3     HOLA    data/raw_videos/HOLA/HOLA_rep_3_110250.mp4   
4     HOLA    data/raw_videos/HOLA/HOLA_rep_1_121616.mp4   
..     ...                                           ...   
393  COMER  data/raw_videos/COMER/COMER_rep_1_115655.mp4   
394  COMER  data/raw_videos/COMER/COMER_rep_3_105753.mp4   
395  COMER  data/raw_videos/COMER/COMER_rep_1_115734.mp4   
396  COMER  data/raw_videos/COMER/COMER_rep_0_105601.mp4   
397  COMER  data/raw_videos/COMER/COMER_rep_2_105717.mp4   

                                  landmarks_file  \
0      data/landmarks/HOLA/HOLA_rep_2_114458.npy   
1      data/landmarks/HOLA/HOLA_rep_1_121412.npy   
2      data/landmarks/HOLA/HOLA_rep_1_110156.n

## 7. Control de Calidad Estricto

Implementamos una clase envargada de realizar un **control de calidad (QC) robusto y realista** sobre las secuencias de *landmarks* ya re-muestreadas a una longitud fija (`target_frames = 60`). El objetivo es **filtrar automáticamente vídeos de baja calidad** antes del entrenamiento, manteniendo únicamente muestras consistentes y bien detectadas.

El criterio central de aceptación es el **`valid_frame_ratio`**, es decir, la proporción de frames que cumplen un mínimo de landmarks válidos. Este criterio es más estable que evaluar frame a frame y se adapta al tipo de seña:
- **Señas de 1 mano:** se exige un ratio mínimo más alto (por defecto 0.80).
- **Señas de 2 manos:** se permite un umbral algo más laxo (por defecto 0.70).

Además del *gate* principal, se calcula un **`quality_score`** continuo (0–1) para análisis y ranking:
1. **Detección (`detection_score`)**: media de landmarks detectados, normalizada según el máximo esperable (54 o 75).
2. **Longitud (`length_score`)**: penaliza secuencias demasiado cortas o largas respecto a los 60 frames objetivo.
3. **Consistencia por manos (`consistency_score`)**: evalúa si el número de manos detectadas concuerda con el tipo de seña y refuerza el `valid_frame_ratio`.

La combinación ponderada prioriza detección y estabilidad temporal, sin ser excesivamente estricta.

Una muestra se **acepta** solo si:
- su `quality_score` supera el umbral global (`quality_threshold`),
- tiene al menos `min_frames`,
- y pasa el *gate* principal basado en `valid_frame_ratio` (con *fallback* compatible para metadatos antiguos).

El método `analyze_dataset`:
- genera un **reporte global y por palabra** (tasa de aceptación, calidad media),
- etiqueta cada muestra como `ACCEPT` o `REJECT`,
- y guarda un **dataset limpio** (`dataset_clean_10words.csv`) listo para entrenamiento.

In [ ]:
class StrictQualityController:
    """
    QC estricto pero realista para landmarks re-muestreados a `target_frames`.
    - Usa `valid_frame_ratio` como gate principal (robusto).
    - Mantiene score compuesto para análisis/ranking.
    """

    def __init__(
        self,
        config,
        target_frames=60,
        min_valid_ratio_one_hand=0.80,
        min_valid_ratio_two_hands=0.70
    ):
        self.config = config
        self.target_frames = target_frames
        self.min_valid_ratio_one_hand = min_valid_ratio_one_hand
        self.min_valid_ratio_two_hands = min_valid_ratio_two_hands

    def expected_landmarks_max(self, row):
        """Máximo esperable según tipo de seña (solo para normalizar detection_score)."""
        return 75 if row['word'] in TWO_HAND_SIGNS else 54

    def _min_valid_ratio(self, row):
        """Umbral por tipo de seña."""
        return self.min_valid_ratio_two_hands if row['word'] in TWO_HAND_SIGNS else self.min_valid_ratio_one_hand

    def calculate_composite_score(self, row):
        # (1) Detección (suave): normaliza respecto a max esperable
        expected_max = self.expected_landmarks_max(row)
        detection_score = min(float(row['avg_landmarks']) / float(expected_max), 1.0)

        # (2) Longitud: alineada con el resample (60)
        optimal_frames = self.target_frames
        length_score = 1.0 - min(abs(float(row['num_frames']) - optimal_frames) / optimal_frames, 1.0)

        # (3) Consistencia por manos
        if row['word'] in TWO_HAND_SIGNS:
            hands_consistency = float(row.get('both_hands_ratio', 0.0))
        else:
            hands_consistency = 1.0 - float(row.get('no_hand_ratio', 1.0))

        # (4) valid_frame_ratio (si existe) refuerza consistencia
        vfr = row.get('valid_frame_ratio', None)
        if vfr is None or (isinstance(vfr, float) and np.isnan(vfr)):
            vfr = hands_consistency
        else:
            vfr = float(vfr)

        consistency_score = 0.5 * hands_consistency + 0.5 * vfr

        composite = 0.5 * detection_score + 0.25 * length_score + 0.25 * consistency_score
        return min(composite, 1.0)

    def _passes_landmarks_gate(self, row):
        """
        Gate principal:
        - si hay valid_frame_ratio -> usarlo
        - si no -> fallback a manos/avg_landmarks (compatibilidad)
        """
        vfr = row.get('valid_frame_ratio', None)
        if vfr is not None and not (isinstance(vfr, float) and np.isnan(vfr)):
            return float(vfr) >= self._min_valid_ratio(row)

        # fallback (por si alguien ejecuta QC con metadata antiguo)
        expected_max = self.expected_landmarks_max(row)
        # exige una fracción razonable del máximo esperable
        return float(row.get('avg_landmarks', 0.0)) >= 0.85 * float(expected_max)

    def analyze_dataset(self, metadata_path='data/metadata/landmarks_metadata.csv',
                        out_path='data/metadata/dataset_clean_10words.csv'):
        df = pd.read_csv(metadata_path)

        df['quality_score'] = df.apply(self.calculate_composite_score, axis=1)

        min_frames = int(self.config.get('min_frames', 30))
        q_thr = float(self.config.get('quality_threshold', 0.85))

        df['status'] = df.apply(
            lambda r: 'ACCEPT'
            if (
                float(r['quality_score']) >= q_thr and
                int(r['num_frames']) >= min_frames and
                self._passes_landmarks_gate(r)
            )
            else 'REJECT',
            axis=1
        )

        total = len(df)
        accepted = int((df['status'] == 'ACCEPT').sum())
        rejected = total - accepted

        print("\n" + "=" * 60)
        print("REPORTE DE CALIDAD (STRICT QC FINAL)")
        print("=" * 60)
        print(f"Total: {total}")
        print(f"✅ Aceptadas: {accepted} ({accepted/total:.1%})")
        print(f"❌ Rechazadas: {rejected} ({rejected/total:.1%})")
        print(f"Calidad promedio: {df['quality_score'].mean():.2%}")

        print("\n" + "-" * 60)
        print("Por palabra:")
        print("-" * 60)

        word_stats = df.groupby('word').agg(
            Calidad=('quality_score', 'mean'),
            Total=('word', 'count'),
            Aceptados=('status', lambda x: int((x == 'ACCEPT').sum()))
        ).round(3)
        word_stats['Tasa_Aceptación'] = (word_stats['Aceptados'] / word_stats['Total'] * 100).round(1)
        word_stats = word_stats.sort_values('Calidad', ascending=False)
        print(word_stats)

        # Guardar limpio
        df_clean = df[df['status'] == 'ACCEPT'].copy()
        Path(out_path).parent.mkdir(parents=True, exist_ok=True)
        df_clean.to_csv(out_path, index=False)

        print("\n" + "=" * 60)
        print(f"Dataset limpio guardado: {out_path}")
        print(f"   Muestras válidas: {len(df_clean)}")
        print("=" * 60)

        return df, df_clean


# Ejecutar QC
qc = StrictQualityController(CONFIG, target_frames=60, min_valid_ratio_one_hand=0.80, min_valid_ratio_two_hands=0.70)
df_all, df_clean = qc.analyze_dataset()
print("QC Controller final definido")


📊 REPORTE DE CALIDAD (STRICT QC FINAL)
Total: 398
✅ Aceptadas: 388 (97.5%)
❌ Rechazadas: 10 (2.5%)
Calidad promedio: 98.07%

------------------------------------------------------------
📋 Por palabra:
------------------------------------------------------------
           Calidad  Total  Aceptados  Tasa_Aceptación
word                                                 
AYUDA        0.986     38         38            100.0
COMER        0.986     40         40            100.0
BANIO        0.985     40         40            100.0
POR_FAVOR    0.985     40         39             97.5
HOLA         0.982     40         40            100.0
NO           0.979     40         38             95.0
GRACIAS      0.978     40         38             95.0
SI           0.978     40         39             97.5
TU           0.975     40         37             92.5
YO           0.975     40         39             97.5

✅ Dataset limpio guardado: data/metadata/dataset_clean_10words.csv
   Muestras válidas: 

## 8. Data Augmentation y Splits


Preparamos el **dataset final para entrenamiento**, separando claramente las fases de *split* y *data augmentation* para **evitar leakage** y mantener coherencia temporal y espacial con el modelo.

La función `resample_to_len` garantiza que **todas las secuencias tengan la misma longitud temporal** (`target_len = 60`), seleccionando frames de forma uniforme. Esto asegura compatibilidad directa con la arquitectura del modelo, independientemente de la duración original del vídeo.

La función `create_splits_from_clean` genera los conjuntos:
- **Train**
- **Validation**
- **Test**

a partir del dataset ya filtrado por calidad (`dataset_clean_10words.csv`).

Características clave:
- **Estratificación por palabra**: cada clase se divide de forma independiente para mantener equilibrio.
- **Split jerárquico**:  
  - 70% → *train*  
  - 30% → *temp* → 15% *val* + 15% *test*
- **Manejo de pocos datos**: si una palabra tiene muy pocas muestras, se asignan directamente a *train*.
- **Garantía de limpieza**: se comprueba explícitamente que **no hay muestras augmentadas en val/test**.

La clase `ConservativeDataAugmenter` aplica **augmentación ligera y realista**, diseñada para mejorar generalización sin distorsionar la seña:

- **Temporal warp**: pequeñas variaciones en la velocidad del gesto.
- **Escalado espacial**: cambios leves de tamaño alrededor del centro de los puntos válidos.
- **Ruido gaussiano**: ruido suave aplicado solo a landmarks existentes.

- **Solo se aplica a TRAIN** (val/test permanecen intactos).
- Las secuencias augmentadas se **re-muestrean de nuevo a 60 frames**.
- Los archivos augmentados se guardan junto a los originales y se marcan explícitamente (`_aug_`).

El resultado es un nuevo archivo:
- `train_split_augmented.csv` → **train final (original + augmentado)**

Al finalizar obtiene:
- **Train**: `train_split_augmented.csv` (originales + augmentación)
- **Validation**: `val_split.csv` (solo originales)
- **Test**: `test_split.csv` (solo originales)

Obtenemos un dataset **listo para la Fase 2 (entrenamiento del modelo)** con garantías de reproducibilidad y sin contaminación entre conjuntos.

In [ ]:
def resample_to_len(seq: np.ndarray, target_len: int = 60) -> np.ndarray:
    """Resample uniforme: (T, P, 3) -> (target_len, P, 3)."""
    T = len(seq)
    if T == 0:
        # asumimos full landmarks por defecto
        return np.zeros((target_len, 75, 3), dtype=np.float32)
    idx = np.linspace(0, T - 1, target_len).astype(int)
    return seq[idx].astype(np.float32)

def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)


# 1) SPLITS DESDE CLEAN (SOLO ORIGINALES)
def create_splits_from_clean(
    clean_csv: str = "data/metadata/dataset_clean_10words.csv",
    out_dir: str = "data/metadata",
    test_size: float = 0.30,      # 70% train, 30% temp
    val_ratio_in_temp: float = 0.50,  # temp -> 15% val, 15% test
    seed: int = 42
):
    """
    Genera:
      - train_split.csv   
      - val_split.csv     
      - test_split.csv   
    """
    df = pd.read_csv(clean_csv)
    ensure_dir(Path(out_dir))

    train_parts, val_parts, test_parts = [], [], []

    for word in sorted(df["word"].unique()):
        wdf = df[df["word"] == word].copy()

        # barajar para estabilidad
        wdf = wdf.sample(frac=1, random_state=seed).reset_index(drop=True)

        # si hay muy pocas muestras, no partir agresivo
        if len(wdf) < 5:
            train_parts.append(wdf)
            continue

        train_w, temp_w = train_test_split(wdf, test_size=test_size, random_state=seed, shuffle=True)
        val_w, test_w = train_test_split(temp_w, test_size=val_ratio_in_temp, random_state=seed, shuffle=True)

        train_parts.append(train_w)
        val_parts.append(val_w)
        test_parts.append(test_w)

    train_df = pd.concat(train_parts, ignore_index=True).sample(frac=1, random_state=seed)
    val_df   = pd.concat(val_parts, ignore_index=True).sample(frac=1, random_state=seed) if len(val_parts) else pd.DataFrame(columns=df.columns)
    test_df  = pd.concat(test_parts, ignore_index=True).sample(frac=1, random_state=seed) if len(test_parts) else pd.DataFrame(columns=df.columns)

    # Garantía: no aug en val/test
    for name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        if "landmarks_file" in split_df.columns:
            aug_ratio = split_df["landmarks_file"].astype(str).str.contains("_aug", na=False).mean()
            print(f"Augment ratio en {name}: {aug_ratio:.3f}")

    train_path = Path(out_dir) / "train_split.csv"
    val_path   = Path(out_dir) / "val_split.csv"
    test_path  = Path(out_dir) / "test_split.csv"

    train_df.to_csv(train_path, index=False)
    val_df.to_csv(val_path, index=False)
    test_df.to_csv(test_path, index=False)

    print("\nSplits creados:")
    print(f"   Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
    print(f"   Guardados en: {out_dir}")

    return train_df, val_df, test_df


# 2) AUGMENTACIÓN SOLO A TRAIN + GUARDAR train_split_augmented.csv
class ConservativeDataAugmenter:
    """
    Augmentación conservadora:
      - temporal warp
      - spatial scale (sobre puntos válidos)
      - gaussian noise (solo puntos válidos)
    IMPORTANTÍSIMO:
      - Se aplica SOLO a TRAIN para evitar leakage.
      - Se re-muestrea a target_frames para consistencia con el modelo.
    """

    @staticmethod
    def temporal_warp(landmarks, factor_range=(0.90, 1.10)):
        factor = np.random.uniform(*factor_range)
        T = len(landmarks)
        new_T = max(20, int(T * factor))
        idx = np.linspace(0, T - 1, new_T).astype(int)
        return landmarks[idx]

    @staticmethod
    def spatial_scale(landmarks, scale_range=(0.95, 1.05)):
        scale = np.random.uniform(*scale_range)
        aug = landmarks.copy()

        # Escala por frame alrededor del centro de puntos no-cero
        for t in range(len(aug)):
            valid = np.any(aug[t] != 0, axis=1)  # (P,)
            if np.any(valid):
                center = np.mean(aug[t][valid], axis=0)
            else:
                center = np.zeros(3, dtype=np.float32)
            aug[t] = center + (aug[t] - center) * scale

        return aug

    @staticmethod
    def gaussian_noise(landmarks, sigma=0.005):
        noise = np.random.normal(0, sigma, landmarks.shape).astype(np.float32)
        mask = np.any(landmarks != 0, axis=2, keepdims=True)  # (T,P,1)
        return landmarks + noise * mask

    def augment_train_split(
        self,
        train_csv: str = "data/metadata/train_split.csv",
        out_csv: str = "data/metadata/train_split_augmented.csv",
        target_frames: int = 60,
        n_aug_per_sample: int = 1,
        seed: int = 42
    ):
        """
        Crea train_split_augmented.csv:
          - Incluye originales + augmentadas
        NO toca val/test.
        """
        rng = np.random.default_rng(seed)
        df_train = pd.read_csv(train_csv).copy()

        augmented_rows = []

        for _, row in tqdm(df_train.iterrows(), total=len(df_train), desc="Augmentando TRAIN"):
            lm_path = Path(row["landmarks_file"])
            if not lm_path.exists():
                # fallback por si el path no incluye "data/"
                lm_path = Path("data") / lm_path
            if not lm_path.exists():
                print(f"⚠️  No existe landmarks_file: {row['landmarks_file']} -> saltando")
                continue

            landmarks = np.load(lm_path).astype(np.float32)  # (T,75,3) o (T,42,3)

            for k in range(n_aug_per_sample):
                aug = landmarks.copy()
                aug = self.temporal_warp(aug)
                aug = self.spatial_scale(aug)
                aug = self.gaussian_noise(aug)

                # mantener longitud fija (por consistencia con entrenamiento)
                aug = resample_to_len(aug, target_frames)

                # Guardar al lado del original
                aug_name = lm_path.stem + f"_aug_{k}.npy"
                aug_path = lm_path.parent / aug_name
                ensure_dir(aug_path.parent)
                np.save(aug_path, aug)

                aug_row = row.copy()
                aug_row["landmarks_file"] = str(aug_path)

                if "landmarks_hands_only_file" in df_train.columns and pd.notna(row.get("landmarks_hands_only_file", np.nan)):
                    # Si quieres mantener hands-only actualizado, aquí lo regeneramos a partir del full:
                    # Si landmarks ya es hands-only, entonces simplemente lo guardamos igual.
                    # Detectamos por shape:
                    if aug.shape[1] == 42:
                        hands_aug = aug
                    else:
                        hands_aug = aug[:, 33:75, :]  # 42 puntos
                    hands_path = Path("data/landmarks_hands_only") / row["word"] / aug_name
                    ensure_dir(hands_path.parent)
                    np.save(hands_path, hands_aug.astype(np.float32))
                    aug_row["landmarks_hands_only_file"] = str(hands_path)

                aug_row["is_augmented"] = True
                aug_row["augmentation_type"] = "temporal_warp+spatial_scale+gaussian_noise"
                augmented_rows.append(aug_row)

        df_train["is_augmented"] = False
        df_train["augmentation_type"] = "original"

        df_aug = pd.DataFrame(augmented_rows)
        df_train_full = pd.concat([df_train, df_aug], ignore_index=True).sample(frac=1, random_state=seed)

        Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
        df_train_full.to_csv(out_csv, index=False)

        print("\nTrain augmentado guardado:")
        print(f"   {out_csv}")
        print(f"   Originales: {len(df_train)} | Augmentadas: {len(df_aug)} | Total: {len(df_train_full)}")

        # Verificación: aug solo en train
        aug_ratio = df_train_full["landmarks_file"].astype(str).str.contains("_aug", na=False).mean()
        print(f"Augment ratio en train_split_augmented: {aug_ratio:.3f}")

        return df_train_full


# 3.1 Crear splits limpios (solo originales)
train_df, val_df, test_df = create_splits_from_clean(
    clean_csv="data/metadata/dataset_clean_10words.csv",
    out_dir="data/metadata",
    test_size=0.30,
    val_ratio_in_temp=0.50,
    seed=42
)

# 3.2 Augment solo train y guardar train_split_augmented.csv (para usar en Fase 2)
augmenter = ConservativeDataAugmenter()
train_full_df = augmenter.augment_train_split(
    train_csv="data/metadata/train_split.csv",
    out_csv="data/metadata/train_split_augmented.csv",
    target_frames=60,
    n_aug_per_sample=1,   
    seed=42
)

# 3.3 Checks finales  
val = pd.read_csv("data/metadata/val_split.csv")
test = pd.read_csv("data/metadata/test_split.csv")
print("\nCHECK FINAL:")
print("Augment en val:", val["landmarks_file"].astype(str).str.contains("_aug", na=False).mean())
print("Augment en test:", test["landmarks_file"].astype(str).str.contains("_aug", na=False).mean())

print("\nListo para Fase 2:")
print(" - Train (original+aug): data/metadata/train_split_augmented.csv")
print(" - Val (solo original):  data/metadata/val_split.csv")
print(" - Test (solo original): data/metadata/test_split.csv")

🔎 Augment ratio en train: 0.000
🔎 Augment ratio en val: 0.000
🔎 Augment ratio en test: 0.000

✅ Splits creados (SOLO ORIGINALES):
   Train: 268 | Val: 60 | Test: 60
   Guardados en: data/metadata


Augmentando TRAIN: 100%|██████████| 268/268 [00:00<00:00, 400.80it/s]


✅ Train augmentado guardado:
   data/metadata/train_split_augmented.csv
   Originales: 268 | Augmentadas: 268 | Total: 536
🔎 Augment ratio en train_split_augmented: 0.500

✅ CHECK FINAL:
Augment en val: 0.0
Augment en test: 0.0

✅ Listo para Fase 2:
 - Train (original+aug): data/metadata/train_split_augmented.csv
 - Val (solo original):  data/metadata/val_split.csv
 - Test (solo original): data/metadata/test_split.csv
